In [2]:
import sys

from brian2 import Hz
import numpy as np 
import pandas as pd 
import os 
import pickle

from brian2 import Hz,mV
from pathlib import Path
REPO_ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(REPO_ROOT / "code"))
from model_unified_ver import run_exp
from model_unified_ver import default_params as params
import utils as utl
from analysis_of_simulation_result import *
from fitting import hill
from make_network import *
av1a1 = [720575940623041549,720575940622894616,720575940626958878,720575940633984924,720575940611137742,720575940627192337]


labial_cluster = pd.read_parquet(REPO_ROOT/'data/labial_cluster_info_v783.parquet')
TPN1 = [720575940623118029, 720575940624967561]

atGRN_cluster = pd.read_parquet(REPO_ROOT/'data/atGRN_cluster_info_v783.parquet')
atgrn_c2g = {}
for t in np.unique(atGRN_cluster.type):
    atgrn_c2g[t] = list(atGRN_cluster.flyid.values[atGRN_cluster.type==t])


In [11]:
params_opt_tarsal = pickle.load(open(REPO_ROOT/'figure4/figure4F-K.MN9_PER_fitting/fitting_result/tarsal/result_av1a1_170.pkl','rb'))
params_opt_labial = pickle.load(open(REPO_ROOT/'figure4/figure4F-K.MN9_PER_fitting/fitting_result/labial/result_av1a1_170.pkl','rb'))

V_l1l2,K_l1l2,n_l1l2,V_l3,K_l3,n_l3,k_act_L,r0_L,h_L = params_opt_labial.x
V_at,K_at,n_at,V_tp,K_tp,n_tp,k_act_T,r0_T,h_T = params_opt_tarsal.x

c_range = [10,50,100,500]

f1_range_atGRN = [0,*np.round(hill(c_range,V_at,K_at,n_at),1)]
f2_range_TPN1 = [0,*np.round(hill(c_range,V_tp,K_tp,n_tp),1)]

f1_range_l1l2 = [0,*np.round(hill(c_range,V_l1l2,K_l1l2,n_l1l2),1)]
f2_range_l3 = [0,*np.round(hill(c_range,V_l3,K_l3,n_l3),1)]

In [10]:
params['w_syn'] = 0.275*mV
params['n_run'] = 100
config = {
    'path_res'  : f'{os.getcwd()}/result_atGRN&TPN1',                              # directory to store results
    'path_comp' : REPO_ROOT / "data" /'Completeness_783.csv',         # csv of the complete list of Flywire neurons
    'path_con'  : REPO_ROOT / "data" / 'Connectivity_783.parquet',   # connectivity data
    'n_proc'    : -1,                                               # number of CPU cores (-1: use all)
}



if 'params' not in os.listdir(f'{config["path_res"]}'):
    os.mkdir(f'{config["path_res"]}/params')

pickle.dump(params,open(f'{config["path_res"]}/params/params.pkl','wb'))


neu_exc = atgrn_c2g['a6']+atgrn_c2g['a7']

neu_add = [TPN1,av1a1]
except_ids = [atgrn_c2g['a6']+atgrn_c2g['a7']+TPN1]
except_wsyn = [0.825*mV]
neu_slnc = []


for f1 in f1_range_atGRN:
    for f2 in f2_range_TPN1:
        for f3 in [0, 170]:
            params['r_poi1'] = f1 * Hz
            params['r_poi2'] = f2 * Hz
            params['r_poi3'] = f3 * Hz

            run_exp(
                exp_name=f'{f1}Hz_{f2}Hz_{f3}Hz',
                neu_exc=neu_exc,
                neu_exc_add=neu_add,
                Except_output_Ids=except_ids,
                Except_output_w_syn=except_wsyn,
                neu_slnc=neu_slnc,
                params=params,
                **config
            )

array([ 33.1,  69.5,  88.4, 127.2])

In [ ]:

params['w_syn'] = 0.275*mV
params['n_run'] = 100
config = {
    'path_res'  : f'{os.getcwd()}/result_L1L2&L3',                              # directory to store results
    'path_comp' : REPO_ROOT / "data" /'Completeness_783.csv',         # csv of the complete list of Flywire neurons
    'path_con'  : REPO_ROOT / "data" / 'Connectivity_783.parquet',   # connectivity data
    'n_proc'    : -1,                                               # number of CPU cores (-1: use all)
}



if 'params' not in os.listdir(f'{config["path_res"]}'):
    os.mkdir(f'{config["path_res"]}/params')

pickle.dump(params,open(f'{config["path_res"]}/params/params.pkl','wb'))


neu_exc = labial_c2g['L1']+labial_c2g['L2']

neu_add = [labial_c2g['L3'],av1a1]
except_ids = None
except_wsyn = None
neu_slnc = []

for f1 in f1_range_l1l2:
    for f2 in f2_range_l3:
        for f3 in [0, 170]:
            params['r_poi1'] = f1 * Hz
            params['r_poi2'] = f2 * Hz
            params['r_poi3'] = f3 * Hz

            run_exp(
                exp_name=f'{f1}Hz_{f2}Hz_{f3}Hz',
                neu_exc=neu_exc,
                neu_exc_add=neu_add,
                Except_output_Ids=except_ids,
                Except_output_w_syn=except_wsyn,
                neu_slnc=neu_slnc,
                params=params,
                **config
            )